# ChatGLM2-6B 的P-Tuning v2 微调
在 ChatGLM2-6B 的代码仓库中，提供了基于 P-Tuning v2 的微调代码。通过这种方式，可以将需要微调的参数量大幅减少到全量参数的 $0.1\%$。此外，结合一些技术手段，比如：
- 模型量化
- 梯度检查点
最低只需要 7GB 显存就可以完成微调。

## 数据爬取
我们从深交所互动易的问答模块中爬取了1000多条问答数据。每个问答对都被整理成以下格式：`{"question": "问题", "answer": "答案"}`，用于微调模型。具体代码如下所示：
```python
data = [
    {"question": "问题1", "answer": "答案1"},
    {"question": "问题2", "answer": "答案2"},
    # 更多数据...
]
```

运行完下面的代码后，程序会在 `DataSet` 目录下生成一个名为 `深交所互动易.json` 的文件。这个文件的数据量不算太大，我们可以手动把它分成两部分：

1. **训练数据**：1000 条数据，保存为 `train.json`。
2. **测试数据**：剩下的数据，保存为 `dev.json`。

这样，数据就被分成了训练集和测试集，方便后续使用。

In [ ]:
import requests
import json
def post_request(url, data):
    headers = {
        "Use  r-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 
(KHTML, like Gecko) Chrome/94.0.4606.71 Safari/537.36",
        "Content-Type": "application/x-www-form-urlencoded"
    }
    response = requests.post(url, headers=headers, data=data)
    if response.status_code == 200:
        return response.json()
    else:
        return None
def get_questions_and_answers():
    base_url = "http://irm.cninfo.com.cn/newircs/index/search"
    para  ms  =  {"pageNo":  1,  "pageSize":  10,  "searchTypes":  "1,11",  "highLight":  
"true"}
    question_list = []
    total_pages = 200
    for page in range(1, total_pages + 1):
        params["pageNo"] = page
        response = post_request(base_url, params)
        if response and "results" in response:
            results = response.get("results", [])
            for result in results:
                question = result.get("mainContent")
                company_name = result.get("companyShortName")
                stock_code = result.get("stockCode")
                answer = result.get("attachedContent")
                if answer:
                    ques  tion_list.append({"question": f" 问{company_name}({stock_code}): {question}", "answer": answer})
        print("Page {} processed".format(page))
    return question_list
def save_to_file(question_list):
    with open("./DataSet/ 深交所互动易.json", "w", encoding="utf-8") as file:
        for item in question_list:
            json.dump(item, file, ensure_ascii=False)
            file.write("\n")
if __name__ == "__main__":
    questions_and_answers = get_questions_and_answers()
    save_to_file(questions_and_answers)

## 模型下载
下载ChatGLM2-6B 的代码仓库，安装相关依赖。为了微调，还需要单独安装一些其他的依赖包。

```bash
git clone https://github.com/THUDM/ChatGLM2-6B
cd ChatGLM2-6B
pip install -r requirements.txt
pip install rouge_chinese nltk jieba datasets
```

## 开始微调
1. 该目录中的代码是官方提供的微调案例实现。
2. 接着，创建一个名为 `finetune_data` 的文件夹。
3. 将 `train.json` 和 `dev.json` 这两个文件放入 `finetune_data` 文件夹中。
4. 最后，编写一个训练脚本，命名为 `finetune.sh`。

```bash
torchrun main.py \
    --do_train \
    --train_file finetune_data/train.json \
    --validation_file finetune_data/dev.json \
    --preprocessing_num_workers 10 \
    --prompt_column question \
    --response_column answer \
    --overwrite_cache \
    --model_name_or_path THUDM/chatglm2-6b \
    --output_dir output/finetune-chatglm2-6b-pt \
    --overwrite_output_dir \
    --max_source_length 128 \
    --max_target_length 512 \
    --per_device_train_batch_size 2 \
    --per_device_eval_batch_size 1 \
    --gradient_accumulation_steps 16 \
    --predict_with_generate \
    --max_steps 2000 \
    --logging_steps 100 \
    --save_steps 100 \
    --learning_rate 2e-2 \
    --pre_seq_len 128 \
    --quantization_bit 4

接下来介绍一下上述部分训练参数的意义。
- `train_file、validation_file`：训练文件和验证文件。
- `prompt_column、response_column`：用于设置模型的输入列和输出列。
- `per_device_train_batch_size、gradient_accumulation_steps`：梯度累计，若每次训练的批量大小为2，每16 轮训练更新一次参数，则等价于每次训练的批量大小为32。
- `max_steps`：最大训练步数。
- `logging_steps`：每当达到该参数值的训练步数则打印一次日志。
- `save_steps`：每当达到该参数值的训练步数则保存一次模型。
- `learning_rate`：微调的学习率。
- `pre_seq_len`：软提示的长度。
- `quantization_bit`：模型参数量化等级，若不配置，则默认采用FP16 的精度进行加载。

在完成 P-Tuning v2 训练后，我们只需要保存 `PrefixEncoder` 部分的参数。所以，在推理阶段，我们需要同时加载两个东西：

1. 原始的 `ChatGLM2-6B` 模型
2. `PrefixEncoder` 的权重

## 开始评估

为了实现这个目标，我们可以创建一个 `evaluate.sh` 脚本，并在里面指定相应的参数。

```bash
CHECKPOINT=finetune-chatglm2-6b-pt
STEP=2000
torchrun main.py \
    --do_predict \
    --validation_file finetune_data/dev.json \
    --test_file finetune_data/dev.json \
    --overwrite_cache \
    --prompt_column question \
    --response_column answer \
    --model_name_or_path THUDM/chatglm2-6b \
    --ptuning_checkpoint ./output/$CHECKPOINT/checkpoint-$STEP \
    --output_dir ./output/$CHECKPOINT \
    --overwrite_output_dir \
    --max_source_length 128 \
    --max_target_length 512 \
    --per_device_eval_batch_size 1 \
    --predict_with_generate \
    --pre_seq_len 128 \
    --quantization_bit 4
```